# 모델 B — KNN 사이즈 예측
- **입력**: 키(cm), 몸무게(kg)
- **출력**: 사이즈 (XS / S / M / L / XL)
- **핏 보정**: slim → 한 단계 아래, over → 한 단계 위 (인접 사이즈 확률 ≥ 30% 조건)
- **데이터**: `data/size_dataset.csv` — 495,561개

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

SIZE_CLASSES = ['XS', 'S', 'M', 'L', 'XL', '2XL']
SAVE_PATH    = 'model/model_b_knn.pkl'

In [ ]:
# ── 데이터 로드 ────────────────────────────────────
df = pd.read_csv('data/size_dataset.csv', encoding='utf-8-sig')
print(f'전체 샘플: {len(df):,}개')
print(f'컬럼: {df.columns.tolist()}')
print(f'결측값: {df.isnull().sum().to_dict()}')
df.head()

In [ ]:
# ── 사이즈 분포 ────────────────────────────────────
size_counts = df['size'].value_counts().reindex(['XS','S','M','L','XL'], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 분포 막대그래프
axes[0].bar(size_counts.index, size_counts.values, color=['#aac4de','#3498db','#0f3460','#3498db','#aac4de'])
axes[0].set_title('사이즈별 샘플 수')
axes[0].set_ylabel('샘플 수')
for i, v in enumerate(size_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontsize=9)

# 키/몸무게 산점도
colors = {'XS':'#e8f4fd','S':'#85c1e9','M':'#2980b9','L':'#1a5276','XL':'#0d2137'}
for size in ['XS','S','M','L','XL']:
    sub = df[df['size']==size].sample(min(500, len(df[df['size']==size])))
    axes[1].scatter(sub['height'], sub['weight'], c=colors[size], label=size, alpha=0.5, s=10)
axes[1].set_title('키-몸무게 분포 (사이즈별)')
axes[1].set_xlabel('키 (cm)')
axes[1].set_ylabel('몸무게 (kg)')
axes[1].legend()

plt.tight_layout()
plt.show()
print(size_counts.to_string())

In [ ]:
# ── 전처리 & 학습/검증 분리 ───────────────────────
df_clean = df.dropna()
X = df_clean[['height', 'weight']].values
y = df_clean['size'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'학습: {len(X_train):,}  |  검증: {len(X_test):,}')

In [ ]:
# ── KNN 학습 ───────────────────────────────────────
knn = KNeighborsClassifier(n_neighbors=7, metric='euclidean', weights='distance', n_jobs=-1)
knn.fit(X_train_s, y_train)

y_pred = knn.predict(X_test_s)
acc = accuracy_score(y_test, y_pred)
print(f'검증 정확도: {acc:.4f} ({acc:.2%})')
print()
print(classification_report(y_test, y_pred, target_names=['XS','S','M','L','XL']))

In [ ]:
# ── Confusion Matrix ───────────────────────────────
cm = confusion_matrix(y_test, y_pred, labels=['XS','S','M','L','XL'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['XS','S','M','L','XL'],
            yticklabels=['XS','S','M','L','XL'])
plt.title('사이즈 예측 Confusion Matrix')
plt.ylabel('실제')
plt.xlabel('예측')
plt.tight_layout()
plt.show()

In [ ]:
# ── 핏 보정 예측 함수 ──────────────────────────────
def predict_size(height_cm, weight_kg, fit='regular'):
    x     = scaler.transform([[height_cm, weight_kg]])
    proba = knn.predict_proba(x)[0]
    prob_dict = dict(zip(knn.classes_, proba))
    base = knn.predict(x)[0]
    idx  = SIZE_CLASSES.index(base) if base in SIZE_CLASSES else 3

    if fit == 'over' and idx < len(SIZE_CLASSES) - 1:
        nxt = SIZE_CLASSES[idx + 1]
        if prob_dict.get(nxt, 0) >= 0.3:
            return base, nxt
    elif fit == 'slim' and idx > 0:
        nxt = SIZE_CLASSES[idx - 1]
        if prob_dict.get(nxt, 0) >= 0.3:
            return base, nxt
    return base, base

# ── 예측 예시 ──────────────────────────────────────
print('키 168cm / 60kg 예시:')
for fit in ['slim', 'regular', 'over']:
    base, rec = predict_size(168, 60, fit)
    arrow = f' → {rec}' if base != rec else ''
    print(f'  {fit:8s}: {base}{arrow}')

In [ ]:
# ── 모델 저장 ──────────────────────────────────────
with open(SAVE_PATH, 'wb') as f:
    pickle.dump({'knn': knn, 'scaler': scaler}, f)
print(f'저장 완료: {SAVE_PATH}')